### Step 1: Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

### Step 2: Load the Data

In [ ]:
df = pd.read_csv('/mnt/data/cafe_sales.csv')

### Step 3: Explore the Data

##### display the first 10 rows to see the "messy" data

In [ ]:
df.head(10)

##### check how many rows and columns

In [ ]:
df.shape

##### display basic info about the dataset to identify nulls and incorrect data types

In [ ]:
df.info()

##### check unique values for `Item`, `Payment Method`, and `Location`

In [ ]:
print(df['Item'].unique())
print(df['Payment Method'].unique())
print(df['Location'].unique())

##### check missing values

In [ ]:
df.isnull().sum()

### Step 4: Handle Missing Values

##### Spot strings like "UNKNOWN" and "ERROR" that Pandas doesn't see as nulls yet

In [ ]:
df.isin(['UNKNOWN', 'ERROR']).sum()

##### Use `.replace()` to convert "UNKNOWN" and "ERROR" to np.nan

In [ ]:
df.replace(['UNKNOWN', 'ERROR'], np.nan, inplace=True)

### Step 5:Text Sanitization

##### remove hidden spaces from all string columns

In [ ]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

##### convert all Item names to lowercase for consistency

In [ ]:
df['Item'] = df['Item'].str.lower()

##### use a dictionary to fix any typos in item names if they exist

In [ ]:
corrections = {
    'cofee': 'coffee',
    'sandwiche': 'sandwich'
}

df['Item'] = df['Item'].replace(corrections)

### Step 6: Fix Data Types
##### fix incorrect data types (Quantity, Price Per Unit, Total Spent .....)

In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

##### identify and fix rows where Quantity might be zero or negative

In [ ]:
df = df[df['Quantity'] > 0]

### Step 7: Handle Missing Values (Part 2: Fill NaNs)

##### fill missing `Price Per Unit`

In [ ]:
df['Price Per Unit'].fillna(df['Price Per Unit'].mean(), inplace=True)

##### handle other missing values appropriately (like Payment Method or Location)

In [ ]:
df['Payment Method'].fillna(df['Payment Method'].mode()[0], inplace=True)

In [ ]:
df['Location'].fillna(df['Location'].mode()[0], inplace=True)

### Step 8: Logical Validation

##### create a new column `Expected_Total` by multiplying `Quantity * Price Per Unit`

In [ ]:
df['Expected_Total'] = df['Quantity'] * df['Price Per Unit']

##### filter rows where Total Spent is NOT equal to Expected_Total

In [ ]:
wrong = df[df['Total Spent'] != df['Expected_Total']]
wrong

##### fill missing or incorrect Total Spent values using the Expected_Total calculation.

In [ ]:
df['Total Spent'] = df['Expected_Total']

### Step 9: Handle Dates

##### convert Transaction Date to datetime

In [ ]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

##### identify transactions with dates in the future (if any) and handle them

In [ ]:
df = df[df['Transaction Date'] <= pd.Timestamp.today()]

##### extract "Year" and "Month" into separate columns for analysis

In [ ]:
df['Year'] = df['Transaction Date'].dt.year
df['Month'] = df['Transaction Date'].dt.month

### Step 10: Remove Duplicates

##### check for duplicates

In [ ]:
df.duplicated().sum()

##### drop duplicates if found

In [ ]:
df.drop_duplicates(inplace=True)

### Step 9: Clean Text Data

#####  clean text columns (strip spaces, unify format)

In [ ]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)